# **1° Passo:** Instalação das Bibliotecas

In [1]:
# instalando as bibliotecas
!pip install -q netcdf4 ultraplot cartopy boto3 geobr salem rasterio pyproj geopandas descartes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.7/341.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 

# **2° Passo:** Download de Arquivos Auxiliares



In [2]:
import time
# download do arquivo "utilities_goes16e19.py"
!wget -c https://github.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/raw/main/01_utils/utilities_goes16e19.py

--2026-07-24 18:39:47--  https://github.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/raw/main/01_utils/utilities_goes16e19.py
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/main/01_utils/utilities_goes16e19.py [following]
--2026-07-24 18:39:47--  https://raw.githubusercontent.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/main/01_utils/utilities_goes16e19.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17438 (17K) [text/plain]
Saving to: ‘utilities_goes16e19.py’

utilities_goes16e19 100%[===================>]  17.03K  --.-KB/s    in 0s      

# **3° Passo:** Declarando Funções

In [3]:
#==================================================================================================#
#                   FUNÇÃO QUE FAZ O DOWNLOAD DOS DADOS DE 20S DA AWS
#==================================================================================================#
def download_GLM20s_AWS(yyyymmddhhmnss, goes_number, path_dest):

    ''' Função para baixar os arquivos netcdf de 20s do GLM processado e disponibilizado pela Amazon Web Service (AWS)

    Parâmetros:
               yyyymmddhhmnss (string): ano+mes+dia+hora+minuto+segundo. Exemplo: 20251113033020
               goes_number (string): número do satélite GOES: 16 ou 19
               path_dest (string): nome do direório + nome do arquivo que foi baixado

    Retorna:
            file_name (string): nome do direório + nome do arquivo que foi baixado

    Observação: acesso aos dados
                    GOES-16: https://noaa-goes16.s3.amazonaws.com/index.html#ABI-L2-CMIPF/
                    GOES-19: https://noaa-goes19.s3.amazonaws.com/index.html#ABI-L2-CMIPF/
    '''

    year = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%Y')
    day_of_year = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%j')
    hour = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%H')
    min = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%M')
    seg = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S').strftime('%S')

    # informação do repositório da AMAZON. Exemplo: 'noaa-goes19'
    bucket_name = f'noaa-goes{goes_number}'

    # inicializa o S3 cliente
    s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))

    # estutura do arquivo. Exemplo: "GLM-L2-LCFA/2025/317/03/OR_GLM-L2-LCFA_G19_s20253170330200"
    product_name = "GLM-L2-LCFA"
    prefix = f'{product_name}/{year}/{day_of_year}/{hour}/OR_{product_name}_G{goes_number}_s{year}{day_of_year}{hour}{min}{seg}'

    # pesquisa pelo arquivo no servidor
    s3_result = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix, Delimiter = "/")

    # verifica se o arquivo esta diponível
    #-----------------------------------------------------------------------------------------------------------
    if 'Contents' not in s3_result:

        # se o arquivo não esta disponível
        print(f'No files found for the date: {yyyymmddhhmnss}, Product-{product_name}')
        return -1
    else:

        # se o arquivo existe
        for obj in s3_result['Contents']:
            key = obj['Key']

            # imprime o nome do arquivo
            file_name = key.split('/')[-1].split('.')[0]

        # baixa o arquivo
        if os.path.exists(f'{path_dest}/{file_name}.nc'):
            print(f'File {path_dest}/{file_name}.nc exists')
        else:
            print(f'Downloading file {path_dest}/{file_name}.nc')
            s3_client.download_file(bucket_name, key, f'{path_dest}/{file_name}.nc')

    return f'{file_name}'

#**PARTE 4)**: `Dados da Amazon a cada 20s` - Visualização da Imagem de Vários Arquivos de Relâmpagos num Mapa Interativo

![Texto alternativo](https://github.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/blob/main/04_logos/folium.png?raw=true)

Por brevidade e simplificação no processamento dos dados, utilizaremos os seguintes intervalos de tempo:

1.   `0 e 5min:` flashes que ocorreram entre o horário atual do relógio e 5 minutos antes.

2.   `5 e 10min:` flashes que ocorreram entre 5 minutos antes e 10 minutos antes do horário atual do relógio.

3.   `10 e 15min:`  flashes que ocorreram entre 10 minutos antes e 15 minutos antes do horário atual do relógio.

`Em resumo realizaremos o seguinte passo-a-passo:`

1.   **Importação das bibliotecas**
2.   **Criação do diretório de entrada e saída**
3.   **Extrai a data atual do relógio e determina o horário de 15min antes**
4.   **Download dos arquivos do GLM**
5.   **Leitura dos arquivos do GLM**
6.   **Separa os dados de Flash por classe de tempo (0-5min, 5-10min e 10-15min)**
7.   **Plota figura com o ultraplot**
8.   **Salva link do mapa em formato HTML**

In [4]:
%%time
#========================================================================================================================#
#                                           IMPORTAÇÃO DAS BIBLIOTECAS
#========================================================================================================================#
import folium
import pandas as pd
import numpy as np
import xarray as xr
import glob
from datetime import timedelta, datetime
import os
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import warnings
warnings.filterwarnings('ignore')

#========================================================================================================================#
#                                        CRIA DIRETÓRIO DE ENTRADA E SAÍDA
#========================================================================================================================#
dir_input = "/content/input/Parte_4"; os.makedirs(dir_input, exist_ok=True)
dir_output = "/content/output/Parte_4"; os.makedirs(dir_output, exist_ok=True)

#========================================================================================================================#
#                                          DEFINE A DATA DO ARQUIVO
#========================================================================================================================#
# extrai a data atual do relógio
agora = datetime.now()
anof, mesf, diaf, horf, minf = str(agora.year), str(agora.month).zfill(2), str(agora.day).zfill(2), str(agora.hour).zfill(2), str(agora.minute).zfill(2)

# calcula a data/hora de 15 minutos atrás
data_15min_atras = datetime.now() - timedelta(minutes=15)

# extrai o ano, mês, dia, hora, minuto inicial (ou seja, 15 min antes da data atual)
anoi = str(data_15min_atras.year)
mesi = str(data_15min_atras.month).zfill(2)
diai = str(data_15min_atras.day).zfill(2)
hori = str(data_15min_atras.hour).zfill(2)
mini = str(data_15min_atras.minute).zfill(2)

#========================================================================================================================#
#                                                DOWNLOAD DOS DADOS
#========================================================================================================================#
# coloca a data INICIAL numa string
date_ini = str(datetime(int(anoi),int(mesi),int(diai),int(hori),int(mini)))

# coloca a data FINAL numa string
date_end = str(datetime(int(anof),int(mesf),int(diaf),int(horf),int(minf)))

# data inicial é fixada como a data do loop
date_loop = date_ini

# loop nos arquivos do GLM
while (date_loop < date_end):

    # data
    yyyymmddhhmnss = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%Y%m%d%H%M%S')

    # ano, mes, dia, hora, minuto e segundos
    ano = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%Y')
    mes = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%m')
    dia = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%d')
    hor = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%H')
    min = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%M')
    seg = datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S').strftime('%S')

    # define se GOES-16 ou GOES-19
    start_g19 = datetime(2025,4,7,0,0)
    imagem_atual = datetime.strptime(yyyymmddhhmnss, '%Y%m%d%H%M%S')
    goes_number = '16' if imagem_atual < start_g19 else '19'
    print(f'DOWNLOAD DO ARQUIVO GLM ===>>> {ano}-{mes}-{dia} {hor}:{min}:{seg}')

    # download o arquivo
    file_glm20s = download_GLM20s_AWS(yyyymmddhhmnss, goes_number, dir_input)

    # incrementa a variável the date_loop
    date_loop = str(datetime.strptime(date_loop, '%Y-%m-%d %H:%M:%S') + timedelta(seconds=20))
    print('\n')

#========================================================================================================================#
#                                            LEITURA DOS ARQUIVOS DE FLASHES
#========================================================================================================================#
# lista dos arquivos
files = sorted(glob.glob(f'{dir_input}/*nc'))

# inicializa os arrays de time, latitude e longitude dos flashes
times_flash = []
lats_flash = np.array([], dtype=np.float32)
lons_flash = np.array([], dtype=np.float32)

# loop nos arquivos
for file in files:

    # leitura do arquivo
    glm_20s = xr.open_dataset(file)

    # appenda as times/lats/longs
    times_flash.extend(glm_20s['flash_time_offset_of_first_event'].values)
    lats_flash = np.append(lats_flash, glm_20s['flash_lat'].values)
    lons_flash = np.append(lons_flash, glm_20s['flash_lon'].values)

    # fecha o arquivo
    glm_20s.close()

# coloca os flashes num dataframe
data_flash = {'time': times_flash, 'lat': lats_flash, 'lon': lons_flash}
df_flash = pd.DataFrame(data_flash)

# tranforma a coluna data para índice do dataframe
df_flash.set_index('time', inplace=True)

# ordena o dataframe
df_flash.sort_index(inplace=True)

#========================================================================================================================#
#                                            SEPARA OS DADOS POR CLASSE DE TEMPO
#========================================================================================================================#
# data atual do relógio
dt_data_ultima = datetime(int(anof), int(mesf), int(diaf), int(horf), int(minf), 0)

# 0-5min: intervalo entre (data_ultima - 5 min) e data_ultima
dt_time_interval_end_0_5min = dt_data_ultima
dt_time_interval_start_0_5min = dt_data_ultima - timedelta(minutes=5)
time_1a = str(dt_time_interval_end_0_5min)
time_1b = str(dt_time_interval_start_0_5min)

# 5-10min: intervalo entre (data_ultima - 10 min) e (data_ultima - 5 min)
dt_time_interval_end_5_10min = dt_time_interval_start_0_5min
dt_time_interval_start_5_10min = dt_data_ultima - timedelta(minutes=10)
time_2a = str(dt_time_interval_end_5_10min)
time_2b = str(dt_time_interval_start_5_10min)

# 10-15min: intervalo entre (data_ultima - 15 min) e (data_ultima - 10 min)
dt_time_interval_end_10_15min = dt_time_interval_start_5_10min
dt_time_interval_start_10_15min = dt_data_ultima - timedelta(minutes=15)
time_3a = str(dt_time_interval_end_10_15min)
time_3b = str(dt_time_interval_start_10_15min)

# seleciona os flashe de cada classe de tempo
df_0_to_5min = df_flash[(df_flash.index > time_1b) & (df_flash.index <= time_1a)]
df_5_to_10min = df_flash[(df_flash.index > time_2b) & (df_flash.index <= time_2a)]
df_10_to_15min = df_flash[(df_flash.index > time_3b) & (df_flash.index <= time_3a)]

#========================================================================================================================#
#                                                 PLOTA FIGURA
#========================================================================================================================#
# dicionário com os dados organizados
dados = {'0-5min': (df_0_to_5min, 'red'),
         '5-10min': (df_5_to_10min, 'blue'),
         '10-15min': (df_10_to_15min, 'green')}

# ordem desejada
ordem_desejada = ['10-15min', '5-10min', '0-5min']

# criar o mapa base
mapa = folium.Map(location=[0,-70], zoom_start=3, tiles='cartodbdark_matter')

# criar grupos de camadas na ordem desejada com contagem no nome
for nome_horario in ordem_desejada:

    # extrai o dataframe e a cor
    df, cor = dados[nome_horario]

    # calcula o total de raios
    total_raios = len(df)

    # cria o nome com contagem
    nome_com_contagem = f'{nome_horario} = {total_raios} Flashes'

    # adiciona o nome do grupo
    grupo = folium.FeatureGroup(name=nome_com_contagem)

    # loop dos flashes daquel intervalo de tempo
    for idx, row in df.iterrows():
        folium.CircleMarker(location=[row["lat"], row["lon"]],
                            radius=3,
                            color=cor,
                            fill=True,
                            fillColor=cor,
                            fillOpacity=0.8,
                            weight=1).add_to(grupo)

    grupo.add_to(mapa)

# adicionar controle de camadas
folium.LayerControl().add_to(mapa)

# cria a data em formato datetime em UTC
data_utc = datetime(int(anof), int(mesf), int(diaf), int(horf), int(minf))

# converte para horário de Brasília (UTC-3)
data_brasilia = data_utc - timedelta(hours=3)

# formata as datas
data_utc_str = data_utc.strftime("%d/%m/%Y %H:%M") + " UTC"
data_brasilia_str = data_brasilia.strftime("%d/%m/%Y %H:%M") + " BRT (Brasília)"

# título com data da última atualização em ambos os horários (UTC e hora local)
title_html = f'''
             <div style="position: relative; text-align: center;">
                 <h3 style="font-size:20px; margin-bottom:2px;"><b>Mapa de Flashes do GLM (GOES-19) para 3 Intervalos de Horários</b></h3>
                 <p style="font-size:14px; color: #888; margin-top:0px;">
                     <i>Última atualização: {data_brasilia_str}</i><br>
                     <span style="font-size:12px;">({data_utc_str})</span>
                 </p>
             </div>
             '''
mapa.get_root().html.add_child(folium.Element(title_html))

# mostra mapa na tela
mapa

Output hidden; open in https://colab.research.google.com to view.

In [5]:
# salva como arquivo HTML
mapa.save(f'{dir_output}/Parte_4_G19_mapa_interativo_flashes.html')